In [1]:
import sys
import warnings
import numpy as np
import pathlib as pl
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
root_folder = pl.Path.cwd().parents[2]
sys.path.insert(0, str(root_folder / "utilities"))
import common_functions as cf

warnings.filterwarnings('ignore')

initial_data_folder = "data/initial_data/function_8"
initial_inputs_path = pl.Path.joinpath(root_folder, initial_data_folder,  "initial_inputs.npy")
initial_outputs_path = pl.Path.joinpath(root_folder, initial_data_folder, "initial_outputs.npy")

In [2]:
data_in = np.load(initial_inputs_path)
data_out = np.load(initial_outputs_path)
#print(data_in)
#print(data_out)

Week-01

In [3]:
X_init = data_in
y_init = data_out

bounds = np.array([(0.0, 1.0)] * X_init.shape[1])  # assume normalized [0,1] per dim
dim = X_init.shape[1]
y_best = y_init.max()

kernel = Matern(length_scale=np.ones(dim),
                length_scale_bounds=(1e-2, 1e2),
                nu=2.5)
# alpha = noise variance. If outputs are noisy, increase alpha (e.g. 1e-4 or tuned)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, n_restarts_optimizer=5, random_state=0)
gp.fit(X_init, y_init)


# -------------------------
# Expected Improvement (EI)
# -------------------------
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.atleast_2d(x)
    mu, sigma = gp.predict(x, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei  # note: maximize this

# Negative EI for minimizers
def neg_ei(x, gp, y_best, xi=0.01):
    return -expected_improvement(x.reshape(1, -1), gp, y_best, xi=xi)[0]

# -------------------------
# Acquisition optimization: many random starts + DE fallback
# -------------------------
def propose_location(gp, y_best, bounds, n_restarts=40):
    dim = bounds.shape[0]
    best_x = None
    best_val = 1e20

    # Random-start L-BFGS-B
    for _ in range(n_restarts):
        x0 = np.random.uniform(bounds[:,0], bounds[:,1])
        res = minimize(fun=neg_ei,
                       x0=x0,
                       args=(gp, y_best),
                       bounds=bounds,
                       method='L-BFGS-B',
                       options={'maxiter':200})
        if res.fun < best_val:
            best_val = res.fun
            best_x = res.x

    # Try differential evolution as a global check (optional, slower)
    try:
        de_res = differential_evolution(lambda x: neg_ei(x, gp, y_best),
                                       bounds=bounds.tolist(),
                                       maxiter=200, polish=True, seed=0)
        if de_res.fun < best_val:
            best_val = de_res.fun
            best_x = de_res.x
    except Exception:
        pass

    # clamp to bounds and return
    best_x = np.clip(best_x, bounds[:,0], bounds[:,1])
    return best_x

# -------------------------
# If you need a small batch (greedy sequential EI)
# -------------------------
def propose_batch(gp, y_best, bounds, batch_size=3):
    batch = []
    gp_copy = gp
    X_aug = X_init.copy()
    y_aug = y_init.copy()
    for i in range(batch_size):
        x_next = propose_location(gp_copy, y_aug.max(), bounds)
        # "Fake" evaluation: use GP predicted mean (Kriging believer)
        y_fake = gp_copy.predict(x_next.reshape(1, -1))[0]
        # Append to augmented set and refit GP (greedy sequential)
        X_aug = np.vstack([X_aug, x_next])
        y_aug = np.hstack([y_aug, y_fake])
        gp_copy = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
        gp_copy.fit(X_aug, y_aug)
        batch.append(x_next)
    return np.array(batch)

# -------------------------
# Run proposal
# -------------------------
x_next = propose_location(gp, y_best, np.array(bounds))
print("Suggested next point (continuous):", x_next)

# If some dims are categorical/encoded, round or map them here, e.g.:
# x_next[cat_index] = int(np.round(x_next[cat_index] * (num_categories-1)))
#
# Example: If dim 6 encodes optimizer 0..3:
# x_next[6] = int(np.round(x_next[6] * 3))

# -------------------------
# (Optional) propose a small batch
# -------------------------
batch = propose_batch(gp, y_best, np.array(bounds), batch_size=3)
print("Suggested batch of 3 points (continuous):\n", batch)


Suggested next point (continuous): [0.2035016  0.16799932 0.19510813 0.03587436 1.         1.
 0.224367   0.49830431]
Suggested batch of 3 points (continuous):
 [[0.20349986 0.16800217 0.19510467 0.03587474 1.         1.
  0.22436825 0.49809753]
 [0.         0.25829577 0.01129258 0.02401308 1.         1.
  0.         0.35659724]
 [0.1722739  0.         0.29062969 0.         1.         0.
  0.01256004 1.        ]]


Week-02

In [4]:
X_new = np.array([0.141081, 0.102602, 0.176130, 0.000000, 1.000000, 0.409759, 0.147143, 0.000000])
y_new = np.array([9.895763038795])

#append new data
X_all = np.vstack([data_in, X_new])
y_all = np.concatenate([data_out, y_new])

#fit GP surrogate
bounds = np.array([(0.0, 1.0)] * X_all.shape[1])
dim = X_all.shape[1]

kernel = Matern(length_scale=np.ones(dim),
                length_scale_bounds=(1e-2, 1e2),
                nu=2.5)

# Lower alpha for smooth deterministic behavior
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                              normalize_y=True, n_restarts_optimizer=5,
                              random_state=0)
gp.fit(X_all, y_all)

#expected Improvement (EI) — exploitative mode
def expected_improvement(x, gp, y_best, xi=0.001):  # smaller xi = exploit
    x = np.atleast_2d(x)
    mu, sigma = gp.predict(x, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei

def neg_ei(x, gp, y_best, xi=0.001):
    return -expected_improvement(x.reshape(1, -1), gp, y_best, xi=xi)[0]

#Optimize acquisition to find next candidate
def propose_location(gp, y_best, bounds, n_restarts=40):
    best_x, best_val = None, 1e20
    for _ in range(n_restarts):
        x0 = np.random.uniform(bounds[:,0], bounds[:,1])
        res = minimize(fun=neg_ei,
                       x0=x0,
                       args=(gp, y_best, 0.001),
                       bounds=bounds,
                       method='L-BFGS-B',
                       options={'maxiter':200})
        if res.fun < best_val:
            best_val, best_x = res.fun, res.x

    # Optional global search (safety)
    try:
        de_res = differential_evolution(lambda x: neg_ei(x, gp, y_best, 0.001),
                                        bounds=bounds.tolist(),
                                        maxiter=200, polish=True, seed=0)
        if de_res.fun < best_val:
            best_val, best_x = de_res.fun, de_res.x
    except Exception:
        pass

    return np.clip(best_x, bounds[:,0], bounds[:,1])

y_best = y_all.max()

# Get next suggested point
x_next = propose_location(gp, y_best, bounds)

print(cf.format_inputdata(x_next))

0.233242-0.210257-0.185535-0.110137-1.000000-1.000000-0.284278-1.000000
